# 01 — Point-cloud inspection

**Inputs:** one external development PLY, in meters with vertical Z. No model is needed. Install `.[geometry,analysis,visualization]`; launch from the checkout. Set `PLY_PATH` in the input cell. Never use test plants.

RGB is inspected as annotation only when the input is explicitly the segmented MaizeField3D export. Natural-color PLYs are not assumed to have semantic supervision. All outputs are cleared in Git.


In [ ]:
from pathlib import Path
import numpy as np

# Set these public inputs before running all cells. Never supply test plants.
# The optional context is used only by the documented synthetic CI smoke test.
CONTEXT = globals().get("_AGRI3D_NOTEBOOK_CONTEXT", {})
PLY_PATH = CONTEXT.get("ply_path", None)
SHOW_VIEWS = CONTEXT.get("visualize", True)
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise ValueError("Launch this notebook from the checkout or its notebooks directory")
if PLY_PATH is None:
    raise ValueError("Set PLY_PATH to an external development PLY, then run all cells")
PLY_PATH = Path(PLY_PATH)


In [ ]:
from agri3d.io.ply import PLYReader
from agri3d.preprocessing.filtering import CloudFilter
from agri3d.preprocessing.labels import decode_maizefield3d_labels

IS_SEGMENTED_MAIZEFIELD3D = False  # explicitly enable only for annotated exports
raw = PLYReader(PLY_PATH).load_point_cloud()
cloud = CloudFilter().remove_nan_inf(raw)
print("Raw / finite points:", len(raw.xyz), len(cloud.xyz))
print("XYZ minima (m):", cloud.xyz.min(axis=0))
print("XYZ maxima (m):", cloud.xyz.max(axis=0))
print("Point attributes:", sorted(cloud.attributes))
if CONTEXT.get("annotated", IS_SEGMENTED_MAIZEFIELD3D):
    annotated = decode_maizefield3d_labels(cloud)
    labels, counts = np.unique(annotated.attributes["label"], return_counts=True)
    print("Semantic counts (0 leaf, 1 stalk):", dict(zip(labels.tolist(), counts.tolist())))


## Deterministic inspection view

This bounded display does not change the cloud or sampling contract. Colors, if shown, are annotations/inspection only, never model input. No rendered dataset output is committed.


In [ ]:
if SHOW_VIEWS:
    import matplotlib.pyplot as plt
    ids = np.linspace(0, len(cloud.xyz) - 1, min(5000, len(cloud.xyz)), dtype=int)
    colors = cloud.attributes.get("colors")
    colors = colors[ids] / 255.0 if colors is not None else "tab:green"
    fig = plt.figure(figsize=(7, 6))
    ax = fig.add_subplot(projection="3d")
    ax.scatter(*cloud.xyz[ids].T, c=colors, s=2)
    ax.set(xlabel="X (m)", ylabel="Y (m)", zlabel="Z (m)")
    plt.show()
